<a href="https://colab.research.google.com/github/ibrahimbarghout/robust-ecg-domain-generalization/blob/main/notebooks/04_label_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# PTB-XL Research Project
## 4. Label Construction

#This notebook converts the raw SCP-ECG annotations in PTB-XL
#into five diagnostic target classes:

#- NORM — Normal ECG
#- MI — Myocardial infarction
#- STTC — ST/T changes
#- CD — Conduction disturbance
#- HYP — Hypertrophy

#The resulting targets are multilabel binary indicators.

#Rhythm annotations such as AFIB, PVC, STACH, and SBRAD
#are retained as metadata but are not included in the five
#primary diagnostic targets.

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import pandas as pd
import numpy as np

from ast import literal_eval

In [4]:
PROJECT_PATH = "/content/drive/MyDrive/PTB-XL Research Project"

DATA_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
)

print("Dataset path:")
print(DATA_PATH)

Dataset path:
/content/drive/MyDrive/PTB-XL Research Project/data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3


In [5]:
df = pd.read_csv(
    os.path.join(DATA_PATH, "ptbxl_database.csv"),
    index_col="ecg_id"
)

scp = pd.read_csv(
    os.path.join(DATA_PATH, "scp_statements.csv"),
    index_col=0
)

print("ECG records:", len(df))
print("SCP statements:", len(scp))

ECG records: 21799
SCP statements: 71


In [6]:
### 4.1 Diagnostic Class Mapping

#PTB-XL provides individual SCP-ECG statements. Each diagnostic
#statement is associated with one of five broad diagnostic classes:

#- NORM
#- MI
#- STTC
#- CD
#- HYP

#We map individual SCP statements to these five classes using
#the `diagnostic_class` field provided by the PTB-XL
#`scp_statements.csv` metadata.

#Rhythm statements are not included in these diagnostic targets.

In [7]:
diagnostic_mapping = scp[
    scp["diagnostic"] == 1
][
    ["description", "diagnostic_class", "diagnostic_subclass"]
].copy()

display(diagnostic_mapping)

,description,diagnostic_class,diagnostic_subclass
NDT,non-diagnostic T abnormalities,STTC,STTC
NST_,non-specific ST changes,STTC,NST_
DIG,digitalis-effect,STTC,STTC
LNGQT,long QT-interval,STTC,STTC
NORM,normal ECG,NORM,NORM
IMI,inferior myocardial infarction,MI,IMI
ASMI,anteroseptal myocardial infarction,MI,AMI
LVH,left ventricular hypertrophy,HYP,LVH
LAFB,left anterior fascicular block,CD,LAFB/LPFB
ISC_,non-specific ischemic,STTC,ISC_


In [8]:
### 4.2 Construct Five Diagnostic Targets

In [9]:
target_classes = [
    "NORM",
    "MI",
    "STTC",
    "CD",
    "HYP"
]

print("Target classes:")
print(target_classes)

Target classes:
['NORM', 'MI', 'STTC', 'CD', 'HYP']


In [10]:
def get_diagnostic_classes(scp_codes):
    """
    Convert the SCP codes for one ECG into the corresponding
    broad diagnostic classes.
    """

    codes = literal_eval(scp_codes)

    classes = set()

    for code in codes:
        if code in scp.index:
            diagnostic_class = scp.loc[code, "diagnostic_class"]

            if pd.notna(diagnostic_class):
                classes.add(diagnostic_class)

    return classes

In [11]:
diagnostic_sets = df["scp_codes"].apply(
    get_diagnostic_classes
)

targets = pd.DataFrame(
    0,
    index=df.index,
    columns=target_classes,
    dtype=int
)

for ecg_id, classes in diagnostic_sets.items():
    for diagnostic_class in classes:
        if diagnostic_class in target_classes:
            targets.loc[ecg_id, diagnostic_class] = 1

targets.index.name = "ecg_id"

display(targets.head(10))

,NORM,MI,STTC,CD,HYP
ecg_id,,,,,
1,1,0,0,0,0
2,1,0,0,0,0
3,1,0,0,0,0
4,1,0,0,0,0
5,1,0,0,0,0
6,1,0,0,0,0
7,1,0,0,0,0
8,0,1,0,0,0
9,1,0,0,0,0


In [12]:
### 4.3 Target Distribution

In [13]:
target_counts = targets.sum().sort_values(
    ascending=False
)

target_percentages = (
    target_counts / len(targets) * 100
)

target_summary = pd.DataFrame({
    "positive_count": target_counts,
    "percentage": target_percentages
})

display(target_summary)

,positive_count,percentage
NORM,9514,43.644204
MI,5469,25.088307
STTC,5235,24.014863
CD,4898,22.468921
HYP,2649,12.151934


In [14]:
### 4.4 Multilabel Target Structure

In [15]:
targets_per_ecg = targets.sum(axis=1)

print("Targets per ECG:")
print(targets_per_ecg.value_counts().sort_index())

print(
    "\nECGs with at least one diagnostic target:",
    (targets_per_ecg > 0).sum()
)

print(
    "ECGs with multiple diagnostic targets:",
    (targets_per_ecg > 1).sum()
)

print(
    "Percentage with multiple diagnostic targets:",
    (targets_per_ecg > 1).mean() * 100
)

Targets per ECG:
0      411
1    16244
2     4068
3      919
4      157
Name: count, dtype: int64

ECGs with at least one diagnostic target: 21388
ECGs with multiple diagnostic targets: 5144
Percentage with multiple diagnostic targets: 23.597412725354374


In [16]:
### 4.5 Diagnostic Target Combinations

In [17]:
from collections import Counter

diagnostic_combinations = Counter()

for _, row in targets.iterrows():
    active_classes = tuple(
        class_name
        for class_name in target_classes
        if row[class_name] == 1
    )

    if len(active_classes) == 0:
        combination = "NONE"
    else:
        combination = " + ".join(active_classes)

    diagnostic_combinations[combination] += 1

print("Diagnostic target combinations:\n")

for combination, count in diagnostic_combinations.most_common():
    percentage = count / len(targets) * 100

    print(
        f"{combination:30s} "
        f"{count:6,d} "
        f"({percentage:5.1f}%)"
    )

Diagnostic target combinations:

NORM                            9,069 ( 41.6%)
MI                              2,532 ( 11.6%)
STTC                            2,400 ( 11.0%)
CD                              1,708 (  7.8%)
MI + CD                         1,297 (  5.9%)
STTC + HYP                        781 (  3.6%)
MI + STTC                         599 (  2.7%)
HYP                               535 (  2.5%)
STTC + CD                         471 (  2.2%)
NONE                              411 (  1.9%)
NORM + CD                         407 (  1.9%)
MI + STTC + HYP                   361 (  1.7%)
CD + HYP                          300 (  1.4%)
MI + STTC + CD                    223 (  1.0%)
STTC + CD + HYP                   211 (  1.0%)
MI + HYP                          183 (  0.8%)
MI + STTC + CD + HYP              156 (  0.7%)
MI + CD + HYP                     117 (  0.5%)
NORM + STTC                        28 (  0.1%)
NORM + STTC + CD                    5 (  0.0%)
NORM + CD + HYP            

In [18]:
### 4.6 Define Abnormal and Reference ECGs

#For this project, an ECG is considered a reference/normal ECG
#when it belongs exclusively to the NORM diagnostic class.

#An ECG is considered abnormal when it contains at least one
#of the four abnormal diagnostic classes:

#MI, STTC, CD, or HYP.

In [19]:
targets["ABNORMAL"] = (
    targets[
        ["MI", "STTC", "CD", "HYP"]
    ].sum(axis=1) > 0
).astype(int)

targets["REFERENCE_NORMAL"] = (
    targets["ABNORMAL"] == 0
).astype(int)

print("Abnormal ECGs:")
print(targets["ABNORMAL"].value_counts())

print("\nReference/normal ECGs:")
print(targets["REFERENCE_NORMAL"].value_counts())

Abnormal ECGs:
ABNORMAL
1    12319
0     9480
Name: count, dtype: int64

Reference/normal ECGs:
REFERENCE_NORMAL
0    12319
1     9480
Name: count, dtype: int64


In [20]:
### 4.7 Label Consistency Checks

In [21]:
print("Total ECGs:", len(targets))

print(
    "Abnormal + Reference:",
    targets["ABNORMAL"].sum()
    + targets["REFERENCE_NORMAL"].sum()
)

print(
    "\nECGs classified as both abnormal and reference:",
    (
        (targets["ABNORMAL"] == 1)
        &
        (targets["REFERENCE_NORMAL"] == 1)
    ).sum()
)

print(
    "ECGs classified as neither:",
    (
        (targets["ABNORMAL"] == 0)
        &
        (targets["REFERENCE_NORMAL"] == 0)
    ).sum()
)

Total ECGs: 21799
Abnormal + Reference: 21799

ECGs classified as both abnormal and reference: 0
ECGs classified as neither: 0


In [22]:
### 4.8 Save Constructed Targets

#The constructed target matrix is saved as a CSV file so that
#subsequent notebooks use the exact same labels without
#reconstructing them independently.

In [23]:
RESULTS_PATH = os.path.join(
    PROJECT_PATH,
    "results"
)

os.makedirs(
    RESULTS_PATH,
    exist_ok=True
)

targets_path = os.path.join(
    RESULTS_PATH,
    "ptbxl_diagnostic_targets.csv"
)

targets.to_csv(targets_path)

print("Targets saved to:")
print(targets_path)

print("File exists:", os.path.exists(targets_path))

Targets saved to:
/content/drive/MyDrive/PTB-XL Research Project/results/ptbxl_diagnostic_targets.csv
File exists: True


In [24]:
## 4.9 Label Construction Summary

#The raw PTB-XL SCP-ECG annotations were mapped to five broad
#diagnostic classes using the official `diagnostic_class`
#metadata provided with the dataset.

#The resulting targets are:

#- NORM
#- MI
#- STTC
#- CD
#- HYP

#Each ECG may have multiple positive diagnostic classes,
#making the downstream task multilabel classification.

#Rhythm annotations such as AFIB, PVC, STACH, and SBRAD are
#not included in the five primary diagnostic targets.

#An additional binary `ABNORMAL` target identifies ECGs with
#at least one abnormal diagnostic class (MI, STTC, CD, or HYP),
#while `REFERENCE_NORMAL` identifies ECGs without any of these
#abnormal classes.

#The final target matrix is saved as:
#`results/ptbxl_diagnostic_targets.csv`.

In [25]:
print([name for name in globals() if not name.startswith("_")])

['In', 'Out', 'get_ipython', 'exit', 'quit', 'drive', 'os', 'pd', 'np', 'literal_eval', 'PROJECT_PATH', 'DATA_PATH', 'df', 'scp', 'diagnostic_mapping', 'target_classes', 'get_diagnostic_classes', 'diagnostic_sets', 'targets', 'ecg_id', 'classes', 'diagnostic_class', 'target_counts', 'target_percentages', 'target_summary', 'targets_per_ecg', 'Counter', 'diagnostic_combinations', 'row', 'active_classes', 'combination', 'count', 'percentage', 'RESULTS_PATH', 'targets_path']


In [26]:
print("Targets shape:", targets.shape)
print("\nTargets:")
display(targets.head())

print("\nTarget file path:")
print(targets_path)

print("\nFile exists:", os.path.isfile(targets_path))

Targets shape: (21799, 7)

Targets:


,NORM,MI,STTC,CD,HYP,ABNORMAL,REFERENCE_NORMAL
ecg_id,,,,,,,
1,1,0,0,0,0,0,1
2,1,0,0,0,0,0,1
3,1,0,0,0,0,0,1
4,1,0,0,0,0,0,1
5,1,0,0,0,0,0,1



Target file path:
/content/drive/MyDrive/PTB-XL Research Project/results/ptbxl_diagnostic_targets.csv

File exists: True
